# Nhiệm vụ 13 — Báo cáo Tổng kết và Đóng gói Bàn giao Milestone M3 (M3 Handoff Package)

Theo quy định tại **Phần 13** trong `M2/Ke_hoach_M2_Phan_cum_co_phieu.md` và các nguyên tắc ranh giới tại `docs/DELTA_UNIFIED_PROJECT_PLAN.md`.

**Mục tiêu chính:**
1. Tiếp nhận kết quả kiểm toán phương pháp luận từ Nhiệm vụ 12.
2. Đóng gói bộ 5 thành phần bàn giao chuẩn hóa vào `M2/artifacts/m2-final-handoff-v1/` và `M2/models/final_selected_model/`.
3. Xác lập rào chắn ranh giới bất biến giữa M2 và M3 (Strict One-Way Handoff, Metric Segregation, PIT Execution).
4. Cung cấp dữ liệu nhãn phân cụm 22 snapshots và chân dung kinh tế chuẩn hóa cho thuật toán phân bổ danh mục M3.

In [ ]:
from pathlib import Path
import sys, os, json, glob, hashlib
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'M2/Ke_hoach_M2_Phan_cum_co_phieu.md').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Chạy notebook từ repo hoặc thư mục M2/notebooks.')
print('Repo root:', ROOT)
sys.path.insert(0, str(ROOT / 'src'))

pd.set_option('display.max_columns', 24)
pd.set_option('display.width', 1000)

## 1. Xác thực Rào chắn Kiểm toán Nhiệm vụ 12 (Task 12 Verification Clearance)

Trước khi đóng gói bàn giao sang M3, hệ thống phải xác nhận:
- Mô hình chiến thắng đã được đóng băng: `KMeans_Baseline` với `Global K = 2`.
- Ranh giới phương pháp luận và kiểm định Holdout năm 2026 đã hoàn tất.
- Quy tắc Occam 0.03 đã được nghiệm thu đạt chuẩn PASS và giới hạn tài liệu của PCA đã được ghi nhận.

In [ ]:
verify_report_path = ROOT / 'M2/reports/Bao_cao_M2_Nhiem_vu_12_Verify.md'
assert verify_report_path.is_file(), 'Thiếu Báo cáo Kiểm toán Nhiệm vụ 12!'

decision_path = ROOT / 'M2/artifacts/m2-evaluation/final_method_decision.json'
with open(decision_path, 'r', encoding='utf-8') as f:
    decision_data = json.load(f)

selected_method = decision_data['selected_method']
print(f"Mô hình chiến thắng được chọn: {selected_method}")
print(f"Global K: {decision_data['global_k']}")
assert selected_method == 'KMeans_Baseline', f"Mô hình phải là KMeans_Baseline, nhận được: {selected_method}"
assert decision_data['global_k'] == 2, f"Global K phải là 2, nhận được: {decision_data['global_k']}"
print("=> Điều kiện tiên quyết: HOÀN TOÀN ĐẠT CHUẨN!")

## 2. Đóng gói Nhãn Cụm Lịch sử 22 Snapshots (`m2_cluster_labels_for_m3.csv`)

Tổng hợp nhãn cụm đã căn chỉnh (`aligned_cluster_id`) qua toàn bộ 22 snapshots:
- **15 tháng Development:** 2023-11-30 đến 2025-01-24 (5.615 quan sát).
- **7 tháng Holdout:** 2026-02-27 đến 2026-08-28 (4.624 quan sát).
- **Tổng cộng:** 10.239 lượt phân loại cổ phiếu.

In [ ]:
handoff_dir = ROOT / 'M2/artifacts/m2-final-handoff-v1'
handoff_dir.mkdir(parents=True, exist_ok=True)

# 1. Dev labels
dev_members = pd.read_csv(ROOT / 'M2/artifacts/m2-evaluation-kmeans/cluster_profile_members.csv')
dev_t4 = pd.read_csv(ROOT / 'M2/artifacts/m2-task4-kmeans-baseline-v1/assignments.csv')
ticker_map = dict(zip(dev_t4['security_id'], dev_t4['ticker']))
dev_members['ticker'] = dev_members['security_id'].map(ticker_map)
dev_members['ticker'] = dev_members['ticker'].fillna(dev_members['security_id'].apply(lambda x: x.split(':')[-1]))

dev_labels = pd.DataFrame({
    'snapshot_date': dev_members['snapshot_date'],
    'security_id': dev_members['security_id'],
    'ticker': dev_members['ticker'],
    'cluster_label': dev_members['aligned_cluster_id'],
    'membership_period': 'development'
})

# 2. Holdout labels
holdout_assign = pd.read_csv(ROOT / 'M2/artifacts/m2-final-holdout-v1/assignments.csv')
holdout_labels = pd.DataFrame({
    'snapshot_date': holdout_assign['snapshot_date'],
    'security_id': holdout_assign['security_id'],
    'ticker': holdout_assign['security_id'].apply(lambda x: x.split(':')[-1]),
    'cluster_label': holdout_assign['aligned_cluster_id'],
    'membership_period': 'holdout'
})

combined_labels = pd.concat([dev_labels, holdout_labels], ignore_index=True)
labels_csv_path = handoff_dir / 'm2_cluster_labels_for_m3.csv'
combined_labels.to_csv(labels_csv_path, index=False)

print(f"Đã xuất: {labels_csv_path}")
print(f"Tổng số dòng: {len(combined_labels)} | Số snapshot: {combined_labels['snapshot_date'].nunique()}")
print("\nPhân bố cụm theo từng giai đoạn:")
print(pd.crosstab(combined_labels['membership_period'], combined_labels['cluster_label'], margins=True))

## 3. Đóng gói Chân dung Kinh tế Tâm Cụm (`m2_cluster_profiles_for_m3.csv`)

Cung cấp thông số tâm cụm trung bình trên 8 đặc trưng tài chính gốc qua từng snapshot và trung vị toàn kỳ:
- **Cụm 0 (Leader / High-Beta Momentum):** Beta cao (1.25 - 1.50), Động lượng mạnh (mom_252 ~ 40-65%), Thanh khoản vượt trội (liquidity_21 ~ 200 - 500 tỷ VND/ngày).
- **Cụm 1 (Follower / Market-Representative):** Beta trung bình (0.60 - 0.75), Động lượng vừa phải (mom_252 ~ 15-25%), Thanh khoản phổ thông (liquidity_21 ~ 10 - 25 tỷ VND/ngày).

In [ ]:
dev_prof = pd.read_csv(ROOT / 'M2/artifacts/m2-evaluation-kmeans/cluster_profiles.csv')
holdout_prof = pd.read_csv(ROOT / 'M2/artifacts/m2-final-holdout-v1/cluster_profiles.csv')

feature_cols = ['mom_21', 'mom_63', 'mom_126', 'mom_252', 'vol_63', 'mdd_126', 'beta_126', 'liquidity_21']

prof_dfs = []
for df, period in [(dev_prof, 'development'), (holdout_prof, 'holdout')]:
    sub = pd.DataFrame()
    sub['snapshot_date'] = df['snapshot_date']
    sub['cluster_label'] = df['aligned_cluster_id']
    sub['size'] = df['size']
    sub['size_ratio'] = df['size_ratio']
    for col in feature_cols:
        sub[col] = df[col] if col in df.columns else df[f"{col}_mean"]
    sub['membership_period'] = period
    prof_dfs.append(sub)

combined_profiles = pd.concat(prof_dfs, ignore_index=True)

# Thêm hàng trung vị tổng hợp (Summary Medians)
summary_rows = []
for cluster_id in [0, 1]:
    c_sub_dev = combined_profiles[(combined_profiles['cluster_label'] == cluster_id) & (combined_profiles['membership_period'] == 'development')]
    c_sub_hold = combined_profiles[(combined_profiles['cluster_label'] == cluster_id) & (combined_profiles['membership_period'] == 'holdout')]
    c_sub_all = combined_profiles[combined_profiles['cluster_label'] == cluster_id]
    
    for label, sub_data, period in [
        ('MEDIAN_DEVELOPMENT', c_sub_dev, 'development'),
        ('MEDIAN_HOLDOUT', c_sub_hold, 'holdout'),
        ('MEDIAN_OVERALL_22M', c_sub_all, 'overall')
    ]:
        row = {
            'snapshot_date': label,
            'cluster_label': cluster_id,
            'size': int(round(sub_data['size'].median())),
            'size_ratio': sub_data['size_ratio'].median(),
            'membership_period': period
        }
        for col in feature_cols:
            row[col] = sub_data[col].median()
        summary_rows.append(row)

all_profiles_df = pd.concat([combined_profiles, pd.DataFrame(summary_rows)], ignore_index=True)
profiles_csv_path = handoff_dir / 'm2_cluster_profiles_for_m3.csv'
all_profiles_df.to_csv(profiles_csv_path, index=False)

print(f"Đã xuất: {profiles_csv_path} ({len(all_profiles_df)} dòng)")
display(all_profiles_df[all_profiles_df['snapshot_date'].str.startswith('MEDIAN')])

## 4. Tham chiếu Luân chuyển Danh mục và Chi phí Giao dịch (`m2_transition_turnover_reference.csv`)

Tổng hợp 20 cặp tháng chuyển đổi thực tế (14 cặp Dev + 6 cặp Holdout). Nhóm M3 bắt buộc phải sử dụng dữ liệu này để:
1. Dự báo tỷ lệ tái cơ cấu danh mục tối thiểu hàng tháng (Turnover Rate).
2. Thiết lập mô hình chi phí trượt giá và phí giao dịch (Transaction Costs Modeling).

In [ ]:
dev_temp = pd.read_csv(ROOT / 'M2/artifacts/m2-evaluation-kmeans/temporal_stability.csv')
dev_temp['membership_period'] = 'development'

holdout_temp = pd.read_csv(ROOT / 'M2/artifacts/m2-final-holdout-v1/temporal_stability.csv')
holdout_temp['membership_period'] = 'holdout'

temp_combined = pd.concat([dev_temp, holdout_temp], ignore_index=True)
temp_combined['pair_id'] = [f"PAIR_{i+1:02d}" for i in range(len(temp_combined))]

cols_temp = [
    'pair_id', 'from_date', 'to_date', 'membership_period', 'n_common', 
    'ari', 'nmi', 'persistence_probability', 'migration_rate', 
    'entered_count', 'exited_count', 'status'
]
temp_combined = temp_combined[cols_temp]

turnover_csv_path = handoff_dir / 'm2_transition_turnover_reference.csv'
temp_combined.to_csv(turnover_csv_path, index=False)

print(f"Đã xuất: {turnover_csv_path} ({len(temp_combined)} cặp tháng)")
print(f"Trung vị ARI toàn kỳ: {temp_combined['ari'].median():.4f}")
print(f"Trung vị Tỷ lệ duy trì cụm (Persistence): {temp_combined['persistence_probability'].median():.4f}")
print(f"Trung vị Tỷ lệ chuyển cụm (Migration Rate): {temp_combined['migration_rate'].median():.4f}")
display(temp_combined[['pair_id', 'from_date', 'to_date', 'membership_period', 'ari', 'persistence_probability', 'migration_rate']])

## 5. Đóng băng Thư mục Mô hình 22 Snapshots và Sinh Manifest SHA-256

Đảm bảo toàn bộ 22 mô hình JSON (15 Development + 7 Holdout) được lưu trữ đầy đủ tại `M2/models/final_selected_model/`.
Sinh tệp siêu dữ liệu `m2_to_m3_handoff_manifest.json` ghi nhận mã băm SHA-256 bảo đảm tính bất biến.

In [ ]:
import shutil

models_dir = ROOT / 'M2/models/final_selected_model'
holdout_models_dir = ROOT / 'M2/models/holdout'
models_dir.mkdir(parents=True, exist_ok=True)

# Copy holdout models
for mf in sorted(holdout_models_dir.glob("*.json")):
    shutil.copy2(mf, models_dir / mf.name)

all_model_files = sorted([f for f in models_dir.glob("*.json") if f.name != 'model_file_manifest.json'])
print(f"Tổng số mô hình trong final_selected_model: {len(all_model_files)}")
assert len(all_model_files) == 22, f"Cần đúng 22 model files, tìm thấy: {len(all_model_files)}"

def calc_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

# Model manifest for full 22-snapshot inventory (preserve model_file_manifest.json for freeze_gate)
model_manifest_22m = {
    "milestone": "M2_FINAL_SELECTED_MODEL_22_SNAPSHOTS",
    "algorithm": "KMeans_Baseline",
    "global_k": 2,
    "n_models": len(all_model_files),
    "files": {mf.name: calc_sha(mf) for mf in all_model_files}
}
with open(models_dir / 'model_file_manifest_22m.json', 'w', encoding='utf-8') as f:
    json.dump(model_manifest_22m, f, indent=2)

# Handoff manifest
handoff_manifest = {
    "manifest_version": "1.0.0",
    "pipeline_milestone": "M2_TO_M3_HANDOFF",
    "timestamp": "2026-10-05T08:35:00+07:00",
    "status": "OFFICIALLY_HANDED_OFF",
    "methodology": {
        "selected_algorithm": "KMeans_Baseline",
        "global_k": 2,
        "features": feature_cols,
        "scaling": "RobustScaler per snapshot (Median, IQR)",
        "selection_rationale": "KMeans_Baseline selected via 5-tier evaluation matrix and Occam's Razor (< 0.03 gap, identical temporal stability to PCA, direct economic interpretability on 8 raw features)"
    },
    "coverage": {
        "n_development_snapshots": 15,
        "development_window": ["2023-11-30", "2025-01-24"],
        "systemic_gap": ["2025-02-03", "2026-01-30"],
        "n_holdout_snapshots": 7,
        "holdout_window": ["2026-02-27", "2026-08-28"],
        "total_snapshots": 22,
        "total_transition_pairs": 20,
        "total_member_observations": len(combined_labels),
        "rebalance_dates": list(combined_labels['snapshot_date'].unique())
    },
    "artifacts_sha256": {
        "m2_cluster_labels_for_m3.csv": calc_sha(labels_csv_path),
        "m2_cluster_profiles_for_m3.csv": calc_sha(profiles_csv_path),
        "m2_transition_turnover_reference.csv": calc_sha(turnover_csv_path),
        "final_selected_models_manifest": calc_sha(models_dir / 'model_file_manifest.json'),
        "all_models_22m_manifest": calc_sha(models_dir / 'model_file_manifest_22m.json')
    },
    "verification_audit_reference": {
        "audit_report": "M2/reports/Bao_cao_M2_Nhiem_vu_12_Verify.md",
        "audit_verdict": "METHODOLOGY_CLEARED_PASS",
        "frozen_winner": "KMeans_Baseline",
        "preregistration_limitation_accepted": True
    },
    "m3_boundary_rules": [
        "Strict one-way handoff: M3 must not modify clustering parameters or Global K based on backtest return/Sharpe.",
        "Metric segregation: M2 metrics (Silhouette, DB, CH, ARI, NMI, Migration) are completely decoupled from M3 portfolio metrics (CAGR, Sharpe, MDD).",
        "PIT execution: Snapshot t labels are available at decision_at (month-end close); portfolio trades execute at t+1 open."
    ]
}

manifest_path = handoff_dir / 'm2_to_m3_handoff_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(handoff_manifest, f, indent=2)

print(f"Đã xuất Handoff Manifest: {manifest_path}")

## 6. Kết luận Bàn giao Chính thức sang Milestone M3

Toàn bộ gói bàn giao kỹ thuật của Milestone M2 đã hoàn tất 100%:
- Nhãn phân cụm 22 snapshots: Sẵn sàng cho quy tắc phân bổ danh mục M3.
- Chân dung kinh tế 8 đặc trưng: Minh bạch căn cứ lựa chọn cụm chiến lược.
- Tham chiếu chuyển cụm 20 cặp tháng: Cung cấp cơ sở tính toán turnover và chi phí giao dịch.
- 22 file mô hình đã fit: Bảo đảm khả năng tái lập tuyệt đối.

**Trạng thái bàn giao:** `OFFICIALLY_HANDED_OFF` -> M3 có thể bắt đầu xây dựng danh mục và backtest ngay lập tức!